In [8]:
!pip install azure-ai-projects azure-core

In [11]:
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("New_Open_AI")

endpoint = "https://azure-ai-email-123.services.ai.azure.com/api/projects/Default"

client = AIProjectClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

openai_client = OpenAI(
    api_key=api_key,
    base_url=f"{endpoint}/openai/v1",
)

resume_text = input("Paste Resume Text: ")

response = openai_client.responses.create(
    input=[{"role": "user", "content": resume_text}],
    extra_body={
        "agent_reference": {
            "name": "resume-agent",
            "version": "2",
            "type": "agent_reference"
        }
    },
)
print(response.output_text)


Paste Resume Text: **Name:** Rahul Sharma Email: [rahul.sharma.dev@gmail.com](mailto:rahul.sharma.dev@gmail.com) | Phone: +91 98765 43210 Location: Noida, Uttar Pradesh, India LinkedIn: linkedin.com/in/rahulsharma-dev GitHub: github.com/rahulsharma-dev  ---  ## Objective  Motivated and detail-oriented Computer Science graduate seeking an entry-level software development role where I can apply my programming skills and grow in a dynamic environment.  ---  ## Education  **B.Tech in Computer Science Engineering** Dr. A.P.J. Abdul Kalam Technical University (AKTU), Lucknow 2022 – 2026 CGPA: 8.2/10  ---  ## Technical Skills  * Languages: Python, Java, C++ * Web: HTML, CSS, JavaScript * Frameworks: Flask, Basics of React * Database: MySQL * Tools: Git, GitHub, VS Code * Concepts: Data Structures & Algorithms, OOP  ---  ## Projects  **AI Interview Simulator**  * Built a web-based application that simulates HR and technical interviews * Used Python (Flask) and OpenAI API * Provides feedback an

In [17]:
client = OpenAI(
    api_key=api_key,
    base_url=f"{endpoint}/openai/v1",
)
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='```json\n{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": [\n    "Project Management",\n    "PR",\n    "Leadership"\n  ]\n}\n```', type='output_text', logprobs=[])]


In [22]:
import json
import re

try:
    # Get full text output safely
    raw_text = response.output_text

    # Extract JSON portion (in case extra text exists)
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        raise ValueError("No valid JSON found in response")

    json_string = match.group(0)

    # Parse JSON
    extracted_data = json.loads(json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")

--- SUCCESS: DATA PARSED ---
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [23]:
from azure.storage.blob import BlobServiceClient
from google.colab import userdata
import datetime

blob_conn_str = userdata.get("bcs")

blob_service_client = BlobServiceClient.from_connection_string(blob_conn_str)

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

blob_client = blob_service_client.get_blob_client(
    container="resume-output",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

print("Saved to storage:", file_name)


Saved to storage: resume_20260421_054501.txt
